# Exercise 7.8 — Dynamical Lie Algebra of the Ising Ansatz

**Chapter 7: Quantum Machine Learning** &nbsp;|&nbsp; *Exercises*

---

A hardware ansatz on an open $N$-qubit chain is generated by $\{\hat{X}_j\hat{X}_{j+1}\}_{j=1}^{N-1}$
and $\{\hat{Z}_j\}_{j=1}^{N}$. Two questions follow. Closing this set under commutation at $N=3$ gives
a dynamical Lie algebra of some dimension, and that dimension decides whether the ansatz trains or
sits on a barren plateau as $N$ grows. Both answers are classical computations, available before any
circuit is run.

## 1. Physical Motivation

The barren-plateau results of Section 7.3 assume the circuit is expressive enough to form a unitary
2-design, in which case the gradient variance falls as $\mathcal{O}(4^{-N})$ and training stops. The
escape is to build an ansatz that cannot reach a 2-design, and the dynamical Lie algebra
$\mathfrak{g}$ measures exactly how far it can reach.

Every parametrized circuit $\hat{U}(\boldsymbol{\theta}) = \prod_k e^{-i\theta_k\hat{H}_k}$ is fixed by
its generators $\{\hat{H}_k\}$, and $\mathfrak{g}$ is the smallest Lie algebra containing their
skew-Hermitian counterparts (Eq. 7.15),
$$
\mathfrak{g} = \mathrm{span}\bigl\{i\hat{H}_k,\; [i\hat{H}_i, i\hat{H}_j],\;
[i\hat{H}_i,[i\hat{H}_j,i\hat{H}_k]],\;\ldots\bigr\}.
$$
Entangling generators fill out all of $\mathfrak{su}(2^N)$, where $\dim\mathfrak{g} = 4^N-1$, and the
generic Haar mechanisms apply. Generators with enough symmetry close early, and the gradient variance
is then bounded below by (Eq. 7.16)
$$
\mathrm{Var}\!\left[\frac{\partial C}{\partial\theta_i}\right] \in
\Omega\!\left(\frac{1}{\dim\mathfrak{g}}\right).
$$
The circuit explores a compact group of dimension $\dim\mathfrak{g}$, and Haar measure on a
$d$-dimensional compact manifold concentrates with variance $\sim 1/d$. A polynomial $d$ concentrates
polynomially, which is survivable.

## 2. Mathematical Toolbox

**The generators at $N=3$.** Five Hermitian operators,
$$
\hat{X}_1\hat{X}_2,\quad \hat{X}_2\hat{X}_3,\quad \hat{Z}_1,\quad \hat{Z}_2,\quad \hat{Z}_3 ,
$$
so $\mathfrak{g}$ starts life $5$-dimensional and grows under nested commutators.

**The first two brackets, by hand.** With $[\hat\sigma_x,\hat\sigma_z] = -2i\hat\sigma_y$,
$$
[\hat{X}_1\hat{X}_2,\,\hat{Z}_2] = \hat{X}_1[\hat{X}_2,\hat{Z}_2] = -2i\,\hat{X}_1\hat{Y}_2
\;\propto\; \hat{X}_1\hat{Y}_2 ,
$$
which is new, and feeding it back in,
$$
[\hat{X}_1\hat{Y}_2,\,\hat{Z}_1] = [\hat{X}_1,\hat{Z}_1]\hat{Y}_2 = -2i\,\hat{Y}_1\hat{Y}_2
\;\propto\; \hat{Y}_1\hat{Y}_2 ,
$$
also new. Each bracket of two Pauli strings is proportional to a third Pauli string, so the closure
never leaves the Pauli basis and the whole calculation is a search over which strings are reachable.

**The closure algorithm.** The routine of Exercise 1.4 carries over unchanged. Hold an orthonormal
(Hilbert-Schmidt) basis of the span, bracket every pair, and keep any result with a nonzero component
orthogonal to what is already stored. Stop when a full sweep adds nothing. Since brackets are bilinear,
an orthonormalized spanning set generates the same algebra as the raw elements, so orthogonalizing as
we go costs nothing and keeps the independence test exact.

## 3. Part 1: closing the algebra at $N=3$

In [ ]:
import numpy as np, itertools
from functools import reduce

I2 = np.eye(2, dtype=complex)
X  = np.array([[0, 1], [1, 0]], dtype=complex)
Y  = np.array([[0, -1j], [1j, 0]])
Z  = np.array([[1, 0], [0, -1]], dtype=complex)
PAULI = {'I': I2, 'X': X, 'Y': Y, 'Z': Z}

def kron(*ops):
    return reduce(np.kron, ops)

def pauli_string(N, sites):
    """Pauli string on N qubits; sites maps qubit index (0-based) -> 'X','Y','Z'."""
    return kron(*[PAULI[sites.get(j, 'I')] for j in range(N)])

def ising_generators(N):
    """Skew-Hermitian generators -i H of the open Ising ansatz: X_j X_{j+1} and Z_j."""
    xx = [-1j * pauli_string(N, {j: 'X', j + 1: 'X'}) for j in range(N - 1)]
    z  = [-1j * pauli_string(N, {j: 'Z'}) for j in range(N)]
    return xx + z

def dla(generators, tol=1e-9):
    """Close a set of skew-Hermitian operators under commutation (cf. Exercise 1.4).

    Returns (dimension, orthonormal Hilbert-Schmidt basis of the span).
    """
    basis = []                       # orthonormal, as flattened vectors
    def add(M):
        v = M.flatten().astype(complex)
        for b in basis:              # project out what is already spanned
            v -= np.vdot(b, v) * b
        if np.linalg.norm(v) > tol:
            basis.append(v / np.linalg.norm(v))
            return True
        return False

    ops = [g / np.linalg.norm(g) for g in generators if add(g)]
    frontier = list(ops)
    while frontier:                  # bracket everything against the newest elements
        new = []
        for a in ops:
            for b in frontier:
                c = a @ b - b @ a
                if np.linalg.norm(c) > tol and add(c):
                    new.append(c / np.linalg.norm(c))
        ops.extend(new)
        frontier = new
    return len(basis), basis

dim_g, basis3 = dla(ising_generators(3))
print(f"N = 3:  dim(g) = {dim_g}")
print(f"        N(2N-1) = {3 * (2 * 3 - 1)}")
print(f"        dim su(2^N) = 4^N - 1 = {4**3 - 1}")
assert dim_g == 15

Fifteen, from five generators. The book's value, and it matches $N(2N-1) = 3\cdot5 = 15$ on the nose.
The algebra is a $15$-dimensional slice of the $63$-dimensional $\mathfrak{su}(8)$, so the ansatz
reaches less than a quarter of the directions a generic circuit would.

Because every bracket stays proportional to a Pauli string, we can ask which of the $63$ traceless
strings actually landed in $\mathfrak{g}$.

In [ ]:
def pauli_content(N, basis):
    """Which traceless Pauli strings lie in the span."""
    found = []
    for label in itertools.product('IXYZ', repeat=N):
        if all(c == 'I' for c in label):
            continue
        v = kron(*[PAULI[c] for c in label]).flatten()
        v = v / np.linalg.norm(v)
        for b in basis:
            v -= np.vdot(b, v) * b
        if np.linalg.norm(v) < 1e-8:
            found.append(''.join(label))
    return found

strings = pauli_content(3, basis3)
print(f"{len(strings)} Pauli strings span g:\n")
for row in range(0, len(strings), 5):
    print('   ' + '  '.join(strings[row:row + 5]))
assert len(strings) == 15

## 4. Why $15$ is $\mathfrak{so}(6)$

The exercise states that these generators close to $\mathfrak{so}(2N)$ with
$\dim\mathfrak{g} = N(2N-1)$, and at $N=3$ that reads $\mathfrak{so}(6)$ with $15$ elements. The
Jordan-Wigner transformation shows where the orthogonal algebra comes from. Define $2N$ Majorana
operators
$$
\hat\gamma_{2j-1} = \Bigl(\prod_{k<j}\hat{Z}_k\Bigr)\hat{X}_j , \qquad
\hat\gamma_{2j}   = \Bigl(\prod_{k<j}\hat{Z}_k\Bigr)\hat{Y}_j ,
$$
which are Hermitian and obey $\{\hat\gamma_a,\hat\gamma_b\} = 2\delta_{ab}\hat{I}$. Both families of
generators are Majorana bilinears,
$$
\hat{Z}_j = -i\,\hat\gamma_{2j-1}\hat\gamma_{2j} , \qquad
\hat{X}_j\hat{X}_{j+1} = -i\,\hat\gamma_{2j}\hat\gamma_{2j+1} ,
$$
so the ansatz supplies precisely the nearest-neighbour bilinears $-i\hat\gamma_a\hat\gamma_{a+1}$ for
$a = 1,\ldots,2N-1$. Brackets of bilinears are bilinears, $[\hat\gamma_a\hat\gamma_b,
\hat\gamma_b\hat\gamma_c] \propto \hat\gamma_a\hat\gamma_c$, so the chain of nearest neighbours
connects every pair $(a,b)$ and the closure is the full set $\{-i\hat\gamma_a\hat\gamma_b\}_{a<b}$.
Counting pairs gives
$$
\dim\mathfrak{g} = \binom{2N}{2} = N(2N-1) ,
$$
which is $\dim\mathfrak{so}(2N)$. The quadratic law is a statement about free fermions, and the Ising
ansatz is trainable for the same reason it is classically solvable.

In [ ]:
def majorana(N, a):
    """gamma_a for a = 1..2N, via Jordan-Wigner."""
    j = (a + 1) // 2                        # site 1..N
    sites = {k - 1: 'Z' for k in range(1, j)}
    sites[j - 1] = 'X' if a % 2 == 1 else 'Y'
    return pauli_string(N, sites)

N = 3
g = [majorana(N, a) for a in range(1, 2 * N + 1)]

clifford = all(np.allclose(g[a] @ g[b] + g[b] @ g[a], 2 * np.eye(2**N) * (a == b))
               for a in range(2 * N) for b in range(2 * N))
print(f"Clifford algebra  {{gamma_a, gamma_b}} = 2 delta_ab :  {clifford}")
assert clifford

for j in range(1, N + 1):
    ok = np.allclose(-1j * g[2*j-2] @ g[2*j-1], pauli_string(N, {j - 1: 'Z'}))
    print(f"  -i gamma_{2*j-1} gamma_{2*j} == Z_{j}          :  {ok}")
    assert ok
for j in range(1, N):
    ok = np.allclose(-1j * g[2*j-1] @ g[2*j], pauli_string(N, {j - 1: 'X', j: 'X'}))
    print(f"  -i gamma_{2*j} gamma_{2*j+1} == X_{j} X_{j+1}      :  {ok}")
    assert ok

In [ ]:
# The 15 bilinears -i gamma_a gamma_b (a < b) and the closed DLA span the same subspace.
bilinears = [-1j * g[a] @ g[b] for a in range(2 * N) for b in range(a + 1, 2 * N)]
Mb = np.array([b.flatten() / np.linalg.norm(b) for b in bilinears])
Mg = np.array(basis3)

rank_b     = np.linalg.matrix_rank(Mb, tol=1e-8)
rank_union = np.linalg.matrix_rank(np.vstack([Mg, Mb]), tol=1e-8)

print(f"number of bilinears -i g_a g_b, a < b :  {len(bilinears)}   (= C(6,2))")
print(f"rank of span{{-i g_a g_b}}              :  {rank_b}")
print(f"dim(g) from the closure               :  {len(basis3)}")
print(f"rank of the two spans stacked         :  {rank_union}")
assert rank_b == rank_union == len(basis3) == 15
print("\nThe spans coincide, so g = so(6), dim 15.")

Stacking the two bases and finding the rank unchanged at $15$ is the identification: the closure of
the Ising generators is neither more nor less than the span of all $\binom{6}{2}$ Majorana bilinears,
which is $\mathfrak{so}(6)$.

Running the closure at several $N$ confirms the law directly. At $N=2$ it returns $6$, the answer
already found in Exercise 1.4 for the two-qubit Ising generators.

In [ ]:
print(f"{'N':>3} {'dim(g)':>8} {'N(2N-1)':>9} {'4^N - 1':>9} {'ratio':>10}")
for N in range(2, 6):
    d, _ = dla(ising_generators(N))
    predicted, generic = N * (2 * N - 1), 4**N - 1
    print(f"{N:>3} {d:>8} {predicted:>9} {generic:>9} {d / generic:>10.4f}")
    assert d == predicted, f"closure disagrees with N(2N-1) at N={N}"

print("\nClosure reproduces N(2N-1) at every N tested.")

## 5. Part 2: is the ansatz trainable?

Trainable, and it stays trainable as $N$ grows. With $\dim\mathfrak{g} = N(2N-1)$ growing only
*quadratically*, the bound of Eq. 7.16 reads
$$
\mathrm{Var}\!\left[\frac{\partial C}{\partial\theta_i}\right] \in
\Omega\!\left(\frac{1}{\dim\mathfrak{g}}\right) = \Omega\!\left(\frac{1}{N^2}\right),
$$
instead of the $\mathcal{O}(4^{-N})$ of a generic circuit. A polynomially small gradient is resolvable
with polynomially many shots, and an exponentially small one is not. The restricted algebra keeps the
ansatz away from the 2-design regime that produces barren plateaus, and the ratio
$\dim\mathfrak{g}/(4^N-1)$ in the table above is already collapsing by $N=5$.

In [ ]:
print(f"{'N':>3} {'dim(g) = N(2N-1)':>18} {'1/dim(g)':>12} {'4^N - 1':>12} {'1/(4^N-1)':>12}")
for N in [3, 5, 10, 20, 50, 100]:
    dim_g_N, generic = N * (2 * N - 1), 4**N - 1
    print(f"{N:>3} {dim_g_N:>18} {1 / dim_g_N:>12.3e} {generic:>12.3e} {1 / generic:>12.3e}")

print("\nAt N = 50 the Ising ansatz guarantees a gradient of order 1e-4, while an unstructured")
print("circuit of the same width offers 1e-60, which no shot budget can resolve.")

## Conclusion

Closing $\{\hat{X}_j\hat{X}_{j+1}, \hat{Z}_j\}$ under commutation at $N=3$ returns
$\dim\mathfrak{g} = 15$, which equals $N(2N-1) = 3\cdot5$ and identifies the algebra as
$\mathfrak{so}(6)$. The Jordan-Wigner picture explains the count: the generators are nearest-neighbour
Majorana bilinears, they close on all $\binom{2N}{2}$ bilinears, and nothing more.

Because $N(2N-1)$ is quadratic rather than exponential, the gradient variance is bounded below by
$\Omega(1/N^2)$ and the ansatz is trainable at any width. The same restriction that saves the
trainability caps the expressivity, since $15$ of $63$ directions at $N=3$ means most of
$\mathfrak{su}(8)$ is unreachable, and that is the trade the inductive bias buys. What makes the
diagnosis useful is that $\dim\mathfrak{g}$ is a classical computation on a handful of matrices, done
before any quantum hardware is touched.